---
<a id='6-ejercicio'></a>

Alumna: Sandra Geraldine Castillo Dorantes

## Sesion 7 - Ejercicio Integrador de Cierre

Combina todo lo visto en la sesion: `groupby`, `apply`, `merge` y manejo de NaN.

In [1]:
# ── Datos con imperfecciones reales ─────────────────────────────────────────
import pandas as pd
import numpy as np
from mi_modulo import clasificar_riesgo

cartera_ej = pd.DataFrame({
    'poliza':  ['A01','A02','A03','A04','A05','A06'],
    'nombre':  ['Sofia','Miguel','Laura','Roberto','Carmen','Pedro'],
    'ramo_id': ['R01','R02','R01','R03','R02','R01'],
    'prima':   [3200, 5800, np.nan, 9100, 4100, 2600],
    'siniest': [0, 1, 0, 2, 0, np.nan],
    'suma':    [300_000,500_000,250_000,600_000,400_000,280_000],
})

catalogo_ramos = pd.DataFrame({
    'ramo_id':    ['R01',   'R02',   'R03'],
    'nombre_ramo':['GMM',   'Autos', 'Vida'],
    'tasa':       [0.022,   0.035,   0.018],
})
print(f'Datos cargados: {len(cartera_ej)} polizas')

Datos cargados: 6 polizas


In [2]:
# ── Tarea 1: Detectar y tratar NaN ───────────────────────────────────────────
# Muestra cuantos NaN hay por columna
# Rellena prima con la mediana y siniest con 0

# Tu codigo aqui:
print('---- NaN por columna ----')
print()
print(f"{'Columna':<9}{'Numero de NaN':<}")
print(cartera_ej.isna().sum())

medianas = cartera_ej['prima'].median()
#cartera = cartera_ej.copy()
cartera_ej['prima'] = cartera_ej['prima'].fillna(medianas)
cartera_ej['siniest'] = cartera_ej['siniest'].fillna(0)

---- NaN por columna ----

Columna  Numero de NaN
poliza     0
nombre     0
ramo_id    0
prima      1
siniest    1
suma       0
dtype: int64


In [3]:
# ── Tarea 2: Merge con catalogo de ramos ─────────────────────────────────────
# Haz un left merge de cartera_ej con catalogo_ramos usando 'ramo_id'
# Verifica que el resultado tiene las columnas nombre_ramo y tasa

# Tu codigo aqui:
cartera_merge = pd.merge(cartera_ej, catalogo_ramos, on='ramo_id', how='left')

print(cartera_merge)

  poliza   nombre ramo_id   prima  siniest    suma nombre_ramo   tasa
0    A01    Sofia     R01  3200.0      0.0  300000         GMM  0.022
1    A02   Miguel     R02  5800.0      1.0  500000       Autos  0.035
2    A03    Laura     R01  4100.0      0.0  250000         GMM  0.022
3    A04  Roberto     R03  9100.0      2.0  600000        Vida  0.018
4    A05   Carmen     R02  4100.0      0.0  400000       Autos  0.035
5    A06    Pedro     R01  2600.0      0.0  280000         GMM  0.022


In [4]:
# ── Tarea 3: Columnas derivadas con apply ────────────────────────────────────
# Crea la columna 'riesgo' usando clasificar_riesgo() de mi_modulo
# Crea la columna 'prima_calc' = suma * tasa * 1.16 (usando apply axis=1)

# Tu codigo aqui:
cartera_merge['riesgo'] = cartera_merge['siniest'].apply(clasificar_riesgo)
cartera_merge['prima_calc'] = cartera_merge.apply(lambda fila: fila['suma'] * fila['tasa'] * 1.16, axis = 1)

In [5]:
# ── Tarea 4: Reporte con groupby ────────────────────────────────────────────
# Con groupby + .agg() calcula por nombre_ramo:
# - polizas: count
# - prima_total: sum
# - prima_prom: mean
# - siniest_total: sum
# Agrega columna pct_cartera = prima del ramo / total * 100

# Tu codigo aqui:
reporte = cartera_merge.groupby('nombre_ramo').agg(
    polizas = ('poliza', 'count'),
    prima_total = ('prima', 'sum'),
    prima_prom = ('prima', 'mean'),
    siniest_total = ('siniest', 'sum')
).round(2).reset_index()

reporte['pct_cartera'] = reporte.apply(lambda x: x['prima_total'] / x['polizas'] * 100, axis = 1)

print("---------- REPORTE ----------")
print(reporte)


---------- REPORTE ----------
  nombre_ramo  polizas  prima_total  prima_prom  siniest_total  pct_cartera
0       Autos        2       9900.0      4950.0            1.0     495000.0
1         GMM        3       9900.0      3300.0            0.0     330000.0
2        Vida        1       9100.0      9100.0            2.0     910000.0
